# 04 · Zone demand anomalies: tuning the alerts behind the Driver Ops Portal

The embedded portal tells ops managers when a zone's demand is far from normal for that
hour, so they can reposition drivers. The model is deliberately simple and explainable:

* **Baseline:** the same zone, weekday and hour over the previous 4 weeks
* **Score:** z = (trips − baseline mean) / max(baseline std, √baseline mean, 1), where the
  Poisson floor stops quiet zones from producing huge scores
* **Alert:** |z| ≥ 3, a change of at least 40%, and at least 25 trips now or in the baseline

This notebook checks what the alerts catch and how many an ops manager would get each day,
because a portal that fires 300 alerts a day gets ignored.

In [13]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+png"   # interactive in VS Code + a static image GitHub can display

con = duckdb.connect(str(ROOT / "data" / "warehouse.duckdb"), read_only=True)
pd.options.display.float_format = "{:,.2f}".format

## 1. Alert volume

In [14]:
summary = con.execute("""
    select company, anomaly_type, count(*) as alerts,
           count(*) / count(distinct pickup_date) as alerts_per_alerting_day
    from gold_zone_hour_scored where anomaly_type is not null
    group by all order by company, anomaly_type
""").df()
summary

,company,anomaly_type,alerts,alerts_per_alerting_day
0,Lyft,drop,2745,20.80
1,Lyft,surge,15552,100.99
2,Uber,drop,5901,43.07
3,Uber,surge,16148,104.86


In [15]:
per_day = con.execute("""
    select pickup_date, company, count(*) filter (where anomaly_type is not null) as alerts
    from gold_zone_hour_scored where baseline_weeks = 4
    group by all order by pickup_date
""").df()
px.line(per_day, x="pickup_date", y="alerts", color="company", title="Alerts per day (default rules)",
        labels={"pickup_date": "", "alerts": "Alerts"}).show()

RuntimeError: 
Image export requires the Kaleido package, v1.0.0 or greater,
which can be installed using pip:

    $ pip install --upgrade "kaleido>=1"


## 2. What do the biggest alerts look like?

In [ ]:
top = con.execute("""
    select s.pickup_hour, s.company, z.zone, z.borough, s.trips, round(s.baseline_trips) as baseline_trips,
           s.pct_vs_baseline, s.z_score, s.anomaly_type
    from gold_zone_hour_scored s join dim_zone z on z.zone_id = s.pu_zone_id
    where s.anomaly_type is not null and s.company = 'Uber'
    order by abs(s.z_score) desc limit 20
""").df()
top

,pickup_hour,company,zone,borough,trips,baseline_trips,pct_vs_baseline,z_score,anomaly_type
0,2024-12-30 02:00:00,Uber,Brooklyn Navy Yard,Brooklyn,201,2.00,113.86,150.62,surge
1,2024-12-30 04:00:00,Uber,Brooklyn Navy Yard,Brooklyn,163,1.00,129.40,144.67,surge
2,2025-02-22 05:00:00,Uber,Brooklyn Navy Yard,Brooklyn,139,2.00,91.67,112.27,surge
3,2025-01-01 03:00:00,Uber,Williamsburg (North Side),Brooklyn,621,30.00,19.87,108.40,surge
4,2025-01-01 03:00:00,Uber,Williamsburg (South Side),Brooklyn,599,41.00,13.61,87.14,surge
5,2025-02-22 04:00:00,Uber,Brooklyn Navy Yard,Brooklyn,112,2.00,63.00,83.34,surge
6,2025-01-01 02:00:00,Uber,Williamsburg (North Side),Brooklyn,749,44.00,16.12,81.48,surge
7,2024-12-29 05:00:00,Uber,Brooklyn Navy Yard,Brooklyn,120,2.00,52.33,78.50,surge
8,2025-01-01 03:00:00,Uber,Greenpoint,Brooklyn,514,27.00,18.21,76.17,surge
9,2025-03-02 06:00:00,Uber,Brooklyn Navy Yard,Brooklyn,140,5.00,30.11,63.88,surge


Look the dates up. Expected hits include New Year's Eve and Day, Thanksgiving, holiday
shopping weekends, Madison Square Garden or Barclays events, and storm days. Alerts that
line up with known events are a quick sanity check on precision.

In [ ]:
zone_name, company = top.iloc[0]["zone"], top.iloc[0]["company"]
t0 = pd.Timestamp(top.iloc[0]["pickup_hour"])
series = con.execute(f"""
    select s.pickup_hour, s.trips, s.baseline_trips, s.anomaly_type
    from gold_zone_hour_scored s join dim_zone z on z.zone_id = s.pu_zone_id
    where z.zone = '{zone_name.replace("'", "''")}' and s.company = '{company}'
      and s.pickup_hour between timestamp '{t0 - pd.Timedelta(days=3)}' and timestamp '{t0 + pd.Timedelta(days=2)}'
    order by 1
""").df()
fig = px.line(series, x="pickup_hour", y=["trips", "baseline_trips"], title=f"{zone_name} ({company}): hourly trips vs baseline")
flag = series[series["anomaly_type"].notna()]
fig.add_scatter(x=flag["pickup_hour"], y=flag["trips"], mode="markers", marker=dict(size=10, color="red"), name="alert")
fig.show()

## 3. Threshold tuning: the alert-fatigue trade-off

The product decision is how many alerts per day an ops manager can act on. The table
re-scores every zone-hour at different z thresholds, with the other rules fixed.

In [ ]:
tuning = con.execute("""
    with s as (
        select * from gold_zone_hour_scored
        where baseline_weeks = 4 and greatest(trips, baseline_trips) >= 25 and abs(pct_vs_baseline) >= 0.4
    )
    select company, t.z as z_threshold,
           count(*) filter (where abs(s.z_score) >= t.z) * 1.0 / count(distinct s.pickup_date) as alerts_per_day,
           count(distinct s.pu_zone_id) filter (where abs(s.z_score) >= t.z) as zones_ever_alerted
    from s cross join (select unnest([2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]) as z) t
    group by all order by company, z_threshold
""").df()
px.line(tuning, x="z_threshold", y="alerts_per_day", color="company", markers=True,
        title="Alerts per day vs z threshold", labels={"alerts_per_day": "Alerts / day", "z_threshold": "|z| ≥"}).show()
tuning

,company,z_threshold,alerts_per_day,zones_ever_alerted
0,Lyft,2.00,269.84,223
1,Lyft,2.50,184.01,220
2,Lyft,3.00,118.81,219
3,Lyft,3.50,76.31,217
4,Lyft,4.00,50.64,215
5,Lyft,5.00,25.09,206
6,Lyft,6.00,14.72,192
7,Uber,2.00,257.21,244
8,Uber,2.50,194.21,244
9,Uber,3.00,143.18,244


**Recommendation template:** choose the threshold that keeps alerts per day within what one
ops manager can action (about 10–20 per shift). Expose it as a setting in the portal (the
app's sidebar already does), and measure the share of alerts actioned (from
`data/app_actions/action_log.csv`) as the portal's precision KPI.

## 4. Known limitations

* Zone-hours with **zero** trips aren't in the hourly table, so a drop all the way to zero
  is not scored. A production version would score against a dense zone × hour spine.
* A 4-week baseline adapts slowly to trend changes. The congestion-pricing launch week shows
  CRZ "drops" that are a new normal rather than incidents.
* Completed trips measure *served* demand. Unmet demand (requests nobody accepted) isn't in
  TLC data. Inside Uber or Lyft, the portal would score requests instead.